# Tutorial 1 — fMRI Preprocessing: Detrending & Nuisance Regression

**Learning objectives**
After completing this tutorial you will be able to:
1. Explain why nuisance regression is necessary before MVPA
2. Load fMRIPrep confound files and select appropriate regressors
3. Compute framewise displacement (FD) and flag high-motion volumes
4. Apply polynomial detrending + confound regression using nilearn
5. Evaluate preprocessing quality using TSNR

**Prerequisites** — familiarity with Python, NumPy, and basic NIfTI I/O.

---


## Section 0 — Setup

We import our shared `fmri_helpers` module, which contains pre-written
implementations of all the functions you will practice using below.


In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import image, plotting
from nilearn.maskers import NiftiMasker

warnings.filterwarnings('ignore')

# Add the parent directory to sys.path so Python can find fmri_helpers.py
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from fmri_helpers import (
    load_bold_and_mask, load_confounds, compute_fd,
    clean_bold_run, plot_motion_qc, track_runtime,
    HMP6, HMP24, CONFOUND_STRATEGIES
)

# ── Dataset paths ─────────────────────────────────────────────────────────
SUBJECT       = "sub-1"
RUN           = 1
USER          = os.getenv("USER")
FMRIPREP_ROOT = f"/scratch/alpine/{USER}/haxby2001/derivatives/fmriprep"

BOLD_PATH  = f"{FMRIPREP_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN}_desc-preproc_bold.nii.gz"
MASK_PATH  = f"{FMRIPREP_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN}_desc-brain_mask.nii.gz"
CONF_PATH  = f"{FMRIPREP_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN}_desc-confounds_timeseries.tsv"

print("Files exist?")
for p in [BOLD_PATH, MASK_PATH, CONF_PATH]:
    print(f"  {'✓' if os.path.exists(p) else '✗'}  {os.path.basename(p)}")


---
## Section 1 — Why Do We Need Nuisance Regression?

Raw fMRIPrep output contains variance from many **non-neural** sources:

| Source | Description |
|---|---|
| **Head motion** | Even mm-scale translations cause voxel-to-voxel signal changes |
| **Scanner drift** | Slow B0 field warming creates low-frequency signal ramps |
| **Physiological noise** | Cardiac pulsation and respiration modulate BOLD |
| **Structured noise** | Components identified by ICA-AROMA or aCompCor |

We remove this variance using **ordinary least-squares (OLS) regression**:

$$\mathbf{y}_{\text{clean}} = \mathbf{y}_{\text{raw}} - \mathbf{X}_{\text{nuisance}} (\mathbf{X}_{\text{nuisance}}^\top \mathbf{X}_{\text{nuisance}})^{-1} \mathbf{X}_{\text{nuisance}}^\top \mathbf{y}_{\text{raw}}$$

Where:
- $\mathbf{y}_{\text{raw}}$ is the raw BOLD time series for one voxel
- $\mathbf{X}_{\text{nuisance}}$ is the matrix of nuisance regressors
- $\mathbf{y}_{\text{clean}}$ is the residual after removing nuisance variance

**Key insight:** We are not filtering — we are projecting the data onto
the subspace orthogonal to the nuisance regressors.

### The 24-HMP model (HMP24)
A common choice for task fMRI: 6 head motion parameters (tx, ty, tz, rx, ry, rz)
plus their first derivatives, squared values, and squared derivatives.
This gives 24 regressors that capture motion-related signal including spin-history
effects from prior-volume displacement.


---
## Section 2 — Loading BOLD Data and Brain Mask

`load_bold_and_mask()` wraps nilearn's `load_img` with an optional dummy-scan drop.


In [ ]:
bold_img, mask_img = load_bold_and_mask(BOLD_PATH, MASK_PATH, n_dummy_scans=0)

print(f"BOLD shape   : {bold_img.shape}")
print(f"BOLD voxel   : {bold_img.header.get_zooms()[:3]} mm")
print(f"TR           : {bold_img.header.get_zooms()[3]:.2f} s")
print(f"Mask voxels  : {int(mask_img.get_fdata().astype(bool).sum())}")


### ✏️ Exercise 1 — Explore the BOLD image

Use NiBabel to extract and print:
1. The data matrix for the first time point (volume index 0)  
2. The mean signal across all in-brain voxels at that time point
3. The minimum and maximum signal values within the mask

**Hint:** `image.index_img(bold_img, 0)` extracts a single volume.


In [ ]:
# YOUR CODE HERE
# Step 1: extract volume 0
vol0 = ...

# Step 2: apply mask and get in-brain voxels at t=0
masker_temp = NiftiMasker(mask_img=mask_img, standardize=False, detrend=False)
data_t0     = masker_temp.fit_transform(vol0).ravel()

# Step 3: print summary
print(f"In-brain voxels at t=0 : {data_t0.shape[0]}")
print(f"Mean signal            : {data_t0.mean():.2f}")
print(f"Min / Max signal       : {data_t0.min():.2f} / {data_t0.max():.2f}")


> 💡 **Answer key** at the bottom of this notebook.


---
## Section 3 — Loading and Selecting Confound Regressors

fMRIPrep produces a TSV file with many potential confound columns.
`load_confounds()` selects the appropriate columns based on the requested strategy
and optionally adds binary **spike regressors** for high-motion volumes.

### Available strategies

| Strategy | Description | n regressors (base) |
|---|---|---|
| `hmp6` | 6 head motion params | 6 |
| `hmp24` | 6 HMPs + derivatives + squares | 24 |
| `compcor5` | HMP6 + top 5 aCompCor components | 11 |

Spike regressors are added on top of whichever strategy is chosen.


In [ ]:
# Load confounds using HMP24 strategy, FD threshold = 0.5 mm
conf_array, conf_cols, fd = load_confounds(CONF_PATH, strategy='hmp24', fd_threshold=0.5)

print(f"Confound matrix shape : {conf_array.shape}")
print(f"Confound columns      : {len(conf_cols)}")
print(f"HMP24 columns found   : {sum(1 for c in conf_cols if not c.startswith('spike'))}")
print(f"Spike regressors      : {sum(1 for c in conf_cols if c.startswith('spike'))}")
print(f"\nFirst 5 column names  : {conf_cols[:5]}")


### Understanding framewise displacement (FD)

FD quantifies how much the participant's head moved between consecutive volumes.
The formula sums the absolute displacements across all 6 degrees of freedom,
converting rotations to mm using an assumed brain radius of 50 mm:

$$\text{FD}_t = |\Delta t_x| + |\Delta t_y| + |\Delta t_z| + 50\cdot(|\Delta r_x| + |\Delta r_y| + |\Delta r_z|)$$

Volumes with FD > 0.5 mm are typically considered high-motion.


In [ ]:
# Compute FD from the HMP columns (first 6 columns of conf_array)
fd_computed = compute_fd(conf_array[:, :6])

print(f"Mean FD : {fd_computed.mean():.3f} mm")
print(f"Max FD  : {fd_computed.max():.3f} mm")
print(f"Volumes > 0.5 mm : {(fd_computed > 0.5).sum()}")

# Plot
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(fd_computed, lw=0.9, color='steelblue', label='FD')
ax.axhline(0.5, color='red', lw=1.2, ls='--', label='FD = 0.5 mm')
ax.set_xlabel('Volume'); ax.set_ylabel('Framewise displacement (mm)')
ax.set_title(f'{SUBJECT} — Run {RUN} — Framewise Displacement')
ax.legend(); plt.tight_layout(); plt.show()


### ✏️ Exercise 2 — Compare confound strategies

Load confounds using the `hmp6` strategy instead of `hmp24` and compare:
1. How many fewer regressors does `hmp6` have?
2. Visualise the `trans_x` column from both strategies on the same plot.
3. Why might we prefer `hmp24` for task fMRI?


In [ ]:
# YOUR CODE HERE
conf_hmp6, cols_hmp6, _ = load_confounds(CONF_PATH, strategy=...)

print(f"HMP6  regressors : {len(cols_hmp6)}")
print(f"HMP24 regressors : {len(conf_cols)}")

# Plot trans_x from both
fig, ax = plt.subplots(figsize=(12, 3))
# ... your plotting code here

plt.show()


---
## Section 4 — Nuisance Regression with `clean_bold_run()`

`clean_bold_run()` does the following in a single OLS step:
1. Extract the 2D BOLD matrix from the NIfTI (n_volumes × n_voxels)
2. Build polynomial drift regressors (default: up to degree 2)
3. Concatenate with confound regressors and z-score each column
4. Regress confounds + drift from BOLD using `nilearn.signal.clean()`

**Why one step?** Performing detrending and confound regression separately
(as two sequential OLS operations) is slightly redundant because the polynomial
drift terms are themselves low-frequency — combining them avoids mild
double-subtraction of variance.


In [ ]:
with track_runtime("nuisance regression"):
    clean_2d, masker, tsnr_before, tsnr_after = clean_bold_run(
        bold_img, mask_img, conf_array, tr=2.5, poly_order=2
    )

print(f"\nCleaned data shape : {clean_2d.shape}")
print(f"TSNR before        : {tsnr_before:.2f}")
print(f"TSNR after         : {tsnr_after:.2f}")
print(f"TSNR improvement   : {tsnr_after - tsnr_before:+.2f}")


### Inspecting the effect on a representative voxel

The power spectrum of a voxel's time series before/after cleaning is a useful
QC check — you should see reduction in low-frequency power (drift removal)
without loss in task-frequency content (~0.005–0.1 Hz for BOLD responses).


In [ ]:
# Re-extract raw 2D data to compare
masker_raw = NiftiMasker(mask_img=mask_img, standardize=False, detrend=False, t_r=2.5)
raw_2d     = masker_raw.fit_transform(bold_img)

# Pick a representative high-signal voxel (highest mean signal in mask)
vox_idx = int(raw_2d.mean(0).argmax())
TR      = 2.5
n_vols  = raw_2d.shape[0]

# Compute power spectra
freqs   = np.fft.rfftfreq(n_vols, d=TR)
ps_raw  = np.abs(np.fft.rfft(raw_2d[:, vox_idx]))**2
ps_cln  = np.abs(np.fft.rfft(clean_2d[:, vox_idx]))**2

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(raw_2d[:, vox_idx],   label='Raw',     alpha=0.7, lw=0.8)
axes[0].plot(clean_2d[:, vox_idx], label='Cleaned', alpha=0.8, lw=0.8)
axes[0].set_xlabel('Volume'); axes[0].set_ylabel('BOLD signal')
axes[0].set_title('Time series — representative voxel'); axes[0].legend()

axes[1].semilogy(freqs[1:], ps_raw[1:],  label='Raw',     alpha=0.7, lw=0.9)
axes[1].semilogy(freqs[1:], ps_cln[1:],  label='Cleaned', alpha=0.8, lw=0.9)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Power (log)')
axes[1].set_title('Power spectrum'); axes[1].legend()

plt.tight_layout(); plt.show()


### ✏️ Exercise 3 — Apply preprocessing to all runs

Extend the single-run cleaning above to loop over all 12 runs.
For each run:
1. Load the BOLD and mask images
2. Load the confounds
3. Apply `clean_bold_run()`
4. Collect the TSNR values before/after

Then call `plot_motion_qc()` to visualise the motion and TSNR summary.

**Hint:** Use `fmri_helpers.get_bold_paths()` to get a list of all BOLD file paths.


In [ ]:
from fmri_helpers import get_bold_paths

# Discover all available runs
bold_paths = get_bold_paths(FMRIPREP_ROOT, SUBJECT, n_runs=12, desc='preproc')
print(f"Found {len(bold_paths)} BOLD files")

fd_per_run   = []
tsnr_before  = []
tsnr_after   = []

for ri, bp in enumerate(bold_paths):
    run = ri + 1
    mp = bp.replace('desc-preproc_bold', 'desc-brain_mask')
    cp = bp.replace('desc-preproc_bold.nii.gz', 'desc-confounds_timeseries.tsv')

    # YOUR CODE HERE
    # bold_img, mask_img = ...
    # conf_array, _, fd  = ...
    # clean_2d, masker, tsb, tsa = ...

    fd_per_run.append(fd)
    tsnr_before.append(tsb)
    tsnr_after.append(tsa)
    print(f"  Run {run:02d}: TSNR {tsb:.1f} → {tsa:.1f}")

# Plot QC summary
fig = plot_motion_qc(fd_per_run, tsnr_before, tsnr_after, subject_id=SUBJECT)
plt.show()


---
## Section 5 — Saving Cleaned NIfTI Files

After cleaning, reconstruct 4-D NIfTIs and save them alongside the fMRIPrep outputs.
The `masker.inverse_transform()` method maps the 2-D cleaned array back to 3-D+time.


In [ ]:
OUTPUT_DIR = f"/scratch/alpine/{USER}/haxby2001/derivatives/preprocessed/{SUBJECT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Example: save just run 1 (the one we cleaned above)
clean_img = masker.inverse_transform(clean_2d)
out_name  = f"{SUBJECT}_task-objectviewing_run-{RUN}_desc-hmp24_bold.nii.gz"
out_path  = os.path.join(OUTPUT_DIR, out_name)
clean_img.to_filename(out_path)
print(f"Saved: {out_path}")
print(f"Shape: {clean_img.shape}")


---
## Summary

| Step | What we did | Why |
|---|---|---|
| Load | `load_bold_and_mask()` | Get 4D BOLD + binary mask |
| Confounds | `load_confounds()` | Select HMP24 + spike regressors |
| FD | `compute_fd()` | Quantify head motion |
| Clean | `clean_bold_run()` | Project out nuisance variance |
| QC | `plot_motion_qc()` | Verify TSNR improvement |
| Save | `masker.inverse_transform()` | Write cleaned NIfTI to disk |

The cleaned NIfTIs feed directly into `beta_estimation.ipynb`.

---


---
## 📖 Answer Key


### Exercise 1 — Explore the BOLD image


In [ ]:
# ANSWER KEY — Exercise 1
vol0        = image.index_img(bold_img, 0)
masker_temp = NiftiMasker(mask_img=mask_img, standardize=False, detrend=False)
data_t0     = masker_temp.fit_transform(vol0).ravel()

print(f"In-brain voxels at t=0 : {data_t0.shape[0]}")
print(f"Mean signal            : {data_t0.mean():.2f}")
print(f"Min / Max signal       : {data_t0.min():.2f} / {data_t0.max():.2f}")


### Exercise 2 — Compare confound strategies


In [ ]:
# ANSWER KEY — Exercise 2
conf_hmp6, cols_hmp6, _ = load_confounds(CONF_PATH, strategy='hmp6')

print(f"HMP6  regressors : {len(cols_hmp6)}")
print(f"HMP24 regressors : {len(conf_cols)}")
print(f"Difference       : {len(conf_cols) - len(cols_hmp6)} extra regressors in HMP24")

# trans_x index in each
tx_hmp24 = conf_array[:, conf_cols.index('trans_x')]
tx_hmp6  = conf_hmp6[:, cols_hmp6.index('trans_x')]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tx_hmp24, label='trans_x (hmp24)', alpha=0.9)
ax.plot(tx_hmp6,  label='trans_x (hmp6)',  alpha=0.7, ls='--')
ax.set_xlabel('Volume'); ax.set_ylabel('Translation (mm)')
ax.set_title('trans_x — same in both strategies')
ax.legend(); plt.tight_layout(); plt.show()

# HMP24 includes squared terms which better capture spin-history artefacts
# from large prior-volume movements, and derivative terms for each parameter.
print("\nWhy HMP24? — The squared parameters capture spin-history artefacts:")
print("  Large movement at t-1 affects signal at t even if motion returns to baseline.")
print("  Derivative terms capture rapid movement during acquisition.")


### Exercise 3 — Apply preprocessing to all runs


In [ ]:
# ANSWER KEY — Exercise 3
from fmri_helpers import get_bold_paths

bold_paths   = get_bold_paths(FMRIPREP_ROOT, SUBJECT, n_runs=12, desc='preproc')
fd_per_run   = []
tsnr_before  = []
tsnr_after   = []

for ri, bp in enumerate(bold_paths):
    run = ri + 1
    mp  = bp.replace('desc-preproc_bold', 'desc-brain_mask')
    cp  = bp.replace('desc-preproc_bold.nii.gz', 'desc-confounds_timeseries.tsv')

    bold_i, mask_i     = load_bold_and_mask(bp, mp)
    conf_i, _, fd_i    = load_confounds(cp, strategy='hmp24')
    _, _, tsb, tsa     = clean_bold_run(bold_i, mask_i, conf_i, tr=2.5)

    fd_per_run.append(fd_i)
    tsnr_before.append(tsb)
    tsnr_after.append(tsa)
    print(f"  Run {run:02d}: TSNR {tsb:.1f} → {tsa:.1f}")

fig = plot_motion_qc(fd_per_run, tsnr_before, tsnr_after, subject_id=SUBJECT)
plt.tight_layout(); plt.show()
